# Check Samples Exceeding Max Length

Identify audio samples that exceed:
- Max audio length: 640,000 samples (40 seconds @ 16kHz)
- Max token length: 448 tokens

In [1]:
import librosa
import pandas as pd
from tqdm import tqdm
from transformers import WhisperProcessor

# Config
MAX_AUDIO_LENGTH = 640000  # 40 seconds @ 16kHz
MAX_TOKEN_LENGTH = 448
SAMPLING_RATE = 16000

CSV_PATH = "E:/Work/My Papers/Dravidian Lang Tech 2026/Dialect based speech processing/Final Dataset/Train/transcripts.csv"
WHISPER_MODEL = "vasista22/whisper-tamil-medium"

In [2]:
# Load processor and data
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL, task="transcribe")
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df)}")

Total samples: 5134


In [3]:
# Check each sample
audio_exceeds = []  # Samples exceeding max audio length
token_exceeds = []  # Samples exceeding max token length

for idx in tqdm(range(len(df)), desc="Checking samples"):
    row = df.iloc[idx]
    audio_path = row['audio_path']
    audio_id = row['name']
    transcription = str(row['transcription'])
    
    # Check audio length
    try:
        waveform, sr = librosa.load(audio_path, sr=SAMPLING_RATE, mono=True)
        audio_len = len(waveform)
        if audio_len > MAX_AUDIO_LENGTH:
            audio_exceeds.append({
                'name': audio_id,
                'audio_path': audio_path,
                'audio_length': audio_len,
                'duration_sec': audio_len / SAMPLING_RATE
            })
    except Exception as e:
        print(f"Error loading {audio_id}: {e}")
        continue
    
    # Check token length
    tokens = processor.tokenizer(transcription, return_tensors='pt')
    token_len = tokens['input_ids'].shape[1]
    if token_len > MAX_TOKEN_LENGTH:
        token_exceeds.append({
            'name': audio_id,
            'transcription': transcription[:100] + '...' if len(transcription) > 100 else transcription,
            'token_length': token_len
        })

Checking samples: 100%|██████████| 5134/5134 [00:05<00:00, 856.55it/s] 


In [4]:
print("\n" + "="*60)
print(f"SAMPLES EXCEEDING MAX AUDIO LENGTH ({MAX_AUDIO_LENGTH} samples = {MAX_AUDIO_LENGTH/SAMPLING_RATE:.1f}s)")
print("="*60)
print(f"Count: {len(audio_exceeds)}")
if audio_exceeds:
    audio_df = pd.DataFrame(audio_exceeds)
    audio_df = audio_df.sort_values('audio_length', ascending=False)
    print(audio_df.to_string(index=False))


SAMPLES EXCEEDING MAX AUDIO LENGTH (640000 samples = 40.0s)
Count: 0


In [5]:
print("\n" + "="*60)
print(f"SAMPLES EXCEEDING MAX TOKEN LENGTH ({MAX_TOKEN_LENGTH} tokens)")
print("="*60)
print(f"Count: {len(token_exceeds)}")
if token_exceeds:
    token_df = pd.DataFrame(token_exceeds)
    token_df = token_df.sort_values('token_length', ascending=False)
    print(token_df.to_string(index=False))


SAMPLES EXCEEDING MAX TOKEN LENGTH (448 tokens)
Count: 29
        name                                                                                           transcription  token_length
SP12_CH_M_60 வீட்டுல கல்யாணம் ரெண்டு நாள் நடக்கும் ரெண்டு நாலுல பையன் வீட்டுல தான் எல்லாமே பாப்பாங்க பொண்ணு எடுத்...           616
  SP6_KG_F_9 வெண்டக்காயில நடுவுல இருக்குறதெல்லான் எடுத்து வச்சம்முலாங்க இதவச்சு ஒரு சட்னி செய்யலானுட்டு ஒரு எடத்த...           587
 SP8_CH_M_40 எல்லாமே எங்களுக்கு கிடைக்கும் அங்க. சீப்பாவும் கிடைக்கும் மெட்டிரியல் நல்லாவும் இருக்கும் இன் ஃபேக்ட...           533
 SP6_KG_F_15 இதுல வந்து தக்காளிய எடுத்து வச்சுறலாமுனுட்டு இருக்குறேன். கத்திரிக்கா வந்து ரெவெண்டு செடியா நட்டலாம்...           519
 SP6_KG_F_37 எப்டி இருக்கீங்கோ? கனடால கொரோனா எல்லாம் கம்மியாயிடுச்சுங்கலானு கேட்டு இருக்காங்கோ. அதுக்கு என்னைய பத...           515
 SP6_KG_F_44 அப்புறம் உங்க வூட்டுக்காரரரு கிட்ட பேசுங்கோ. எல்லாருக்கும் வணக்கமுங்கோ. எப்படி இருக்கிரிங்கோ சாமி என...           510
 SP8_CH_M_28 மோஸ்ட்லி சண